# 📊 Project 11 — AgentLens: Production Observability Agent

**Core Concept:** Full observability — latency, cost, tokens, errors, alerts

### Architecture
Agent Call → AgentLens Wrapper → Metrics Store → Dashboard + Alerts

Install

In [1]:
!pip install -q langchain langchain-groq langchain-core loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00


API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "KEY"

All Setup In One Block

In [3]:
import os
import time
import json
import uuid
from datetime import datetime, timezone
from loguru import logger
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import sys

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")

# ── Metrics Store ────────────────────────────────────────────
class MetricsStore:
    def __init__(self):
        self.traces = []
        self.alerts = []
        self.total_cost = 0.0
        self.total_tokens = 0
        self.total_calls = 0
        self.failed_calls = 0

    def record(self, trace: dict):
        self.traces.append(trace)
        self.total_calls += 1
        self.total_cost += trace.get("cost", 0)
        self.total_tokens += trace.get("tokens_used", 0)
        if not trace.get("success", True):
            self.failed_calls += 1

    def add_alert(self, alert: dict):
        self.alerts.append(alert)
        logger.warning(f"ALERT: {alert['type']} | {alert['message']}")

    def get_stats(self) -> dict:
        if not self.traces:
            return {}

        latencies = [t["latency_ms"] for t in self.traces if "latency_ms" in t]
        costs = [t["cost"] for t in self.traces if "cost" in t]

        return {
            "total_calls": self.total_calls,
            "failed_calls": self.failed_calls,
            "success_rate": round((self.total_calls - self.failed_calls) / self.total_calls * 100, 1),
            "total_cost": round(self.total_cost, 6),
            "total_tokens": self.total_tokens,
            "avg_latency_ms": round(sum(latencies) / len(latencies), 2) if latencies else 0,
            "max_latency_ms": round(max(latencies), 2) if latencies else 0,
            "min_latency_ms": round(min(latencies), 2) if latencies else 0,
            "avg_cost_per_call": round(sum(costs) / len(costs), 6) if costs else 0,
            "total_alerts": len(self.alerts)
        }


# ── Alert Manager ────────────────────────────────────────────
class AlertManager:
    def __init__(self, metrics_store: MetricsStore):
        self.store = metrics_store
        self.thresholds = {
            "latency_ms": 3000,
            "cost_per_call": 0.01,
            "error_rate": 20.0,
            "tokens_per_call": 2000
        }

    def check(self, trace: dict):
        if trace.get("latency_ms", 0) > self.thresholds["latency_ms"]:
            self.store.add_alert({
                "type": "HIGH_LATENCY",
                "message": f"Latency {trace['latency_ms']}ms exceeded threshold {self.thresholds['latency_ms']}ms",
                "trace_id": trace.get("trace_id"),
                "timestamp": datetime.now(timezone.utc).isoformat()
            })

        if trace.get("cost", 0) > self.thresholds["cost_per_call"]:
            self.store.add_alert({
                "type": "HIGH_COST",
                "message": f"Cost ${trace['cost']} exceeded threshold ${self.thresholds['cost_per_call']}",
                "trace_id": trace.get("trace_id"),
                "timestamp": datetime.now(timezone.utc).isoformat()
            })

        if trace.get("tokens_used", 0) > self.thresholds["tokens_per_call"]:
            self.store.add_alert({
                "type": "HIGH_TOKEN_USAGE",
                "message": f"Tokens {trace['tokens_used']} exceeded threshold {self.thresholds['tokens_per_call']}",
                "trace_id": trace.get("trace_id"),
                "timestamp": datetime.now(timezone.utc).isoformat()
            })

        if not trace.get("success", True):
            self.store.add_alert({
                "type": "CALL_FAILED",
                "message": f"Agent call failed: {trace.get('error', 'Unknown error')}",
                "trace_id": trace.get("trace_id"),
                "timestamp": datetime.now(timezone.utc).isoformat()
            })


# ── Tracer ───────────────────────────────────────────────────
class Tracer:
    def __init__(self, metrics_store: MetricsStore, alert_manager: AlertManager):
        self.store = metrics_store
        self.alert_manager = alert_manager

    def trace(self, agent_name: str, task: str,
              response: str, latency_ms: float,
              tokens_used: int, cost: float,
              success: bool = True, error: str = None) -> dict:

        trace = {
            "trace_id": str(uuid.uuid4())[:8],
            "agent_name": agent_name,
            "task": task[:100],
            "response_preview": response[:100] if response else "",
            "latency_ms": round(latency_ms, 2),
            "tokens_used": tokens_used,
            "cost": round(cost, 6),
            "success": success,
            "error": error,
            "timestamp": datetime.now(timezone.utc).isoformat()
        }

        self.store.record(trace)
        self.alert_manager.check(trace)
        logger.info(f"Trace recorded: {trace['trace_id']} | Latency: {latency_ms:.0f}ms | Cost: ${cost:.6f}")
        return trace


# ── Observable Agent ─────────────────────────────────────────
class ObservableAgent:
    def __init__(self, name: str, tracer: Tracer):
        self.name = name
        self.tracer = tracer
        self.llm = ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0.3,
            api_key=os.environ["GROQ_API_KEY"]
        )
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful AI assistant. Answer clearly and concisely."),
            ("human", "{task}")
        ])
        self.chain = self.prompt | self.llm

    def run(self, task: str) -> dict:
        start_time = time.time()
        success = True
        response = ""
        error = None

        try:
            result = self.chain.invoke({"task": task})
            response = result.content
        except Exception as e:
            success = False
            error = str(e)
            response = ""

        latency_ms = (time.time() - start_time) * 1000
        tokens_used = len(task.split()) + len(response.split())
        cost = (tokens_used / 1000) * 0.0008

        trace = self.tracer.trace(
            self.name, task, response,
            latency_ms, tokens_used, cost,
            success, error
        )

        return {
            "task": task,
            "response": response,
            "trace": trace,
            "success": success
        }


# ── Dashboard ────────────────────────────────────────────────
class Dashboard:
    def __init__(self, metrics_store: MetricsStore):
        self.store = metrics_store

    def render(self):
        stats = self.store.get_stats()
        print("\n" + "="*60)
        print("AGENTLENS DASHBOARD")
        print("="*60)
        print(f"Total Calls     : {stats.get('total_calls', 0)}")
        print(f"Failed Calls    : {stats.get('failed_calls', 0)}")
        print(f"Success Rate    : {stats.get('success_rate', 0)}%")
        print(f"Total Cost      : ${stats.get('total_cost', 0)}")
        print(f"Total Tokens    : {stats.get('total_tokens', 0)}")
        print(f"Avg Latency     : {stats.get('avg_latency_ms', 0)}ms")
        print(f"Max Latency     : {stats.get('max_latency_ms', 0)}ms")
        print(f"Min Latency     : {stats.get('min_latency_ms', 0)}ms")
        print(f"Avg Cost/Call   : ${stats.get('avg_cost_per_call', 0)}")
        print(f"Total Alerts    : {stats.get('total_alerts', 0)}")

        if self.store.alerts:
            print("\n--- ALERTS ---")
            for alert in self.store.alerts:
                print(f"  [{alert['type']}] {alert['message']}")

        print("\n--- TRACES ---")
        for trace in self.store.traces[-5:]:
            status = "✅" if trace["success"] else "❌"
            print(f"  {status} [{trace['trace_id']}] {trace['agent_name']:15} | {trace['latency_ms']}ms | ${trace['cost']} | {trace['task'][:40]}")
        print("="*60)

metrics_store = MetricsStore()
alert_manager = AlertManager(metrics_store)
tracer = Tracer(metrics_store, alert_manager)
dashboard = Dashboard(metrics_store)
agent = ObservableAgent("ProductionAgent", tracer)
print("AgentLens system ready")

AgentLens system ready


Run Multiple Agent Calls

In [4]:
print("========== RUNNING AGENT CALLS ==========\n")

tasks = [
    "What is machine learning?",
    "Explain the difference between SQL and NoSQL databases",
    "Write a Python function to calculate fibonacci numbers",
    "What are the SOLID principles in software engineering?",
    "Explain how HTTPS works"
]

for task in tasks:
    result = agent.run(task)
    print(f"Task: {task[:50]}")
    print(f"Success: {result['success']} | Latency: {result['trace']['latency_ms']}ms | Cost: ${result['trace']['cost']}")
    print("---")

========== RUNNING AGENT CALLS ==========

06:22:05 | INFO | Trace recorded: cadbcb35 | Latency: 445ms | Cost: $0.000041
Task: What is machine learning?
Success: True | Latency: 445.0ms | Cost: $4.1e-05
---
06:22:06 | INFO | Trace recorded: 93f09371 | Latency: 1155ms | Cost: $0.000178
Task: Explain the difference between SQL and NoSQL datab
Success: True | Latency: 1154.97ms | Cost: $0.000178
---
06:22:07 | INFO | Trace recorded: 7eb4377a | Latency: 431ms | Cost: $0.000090
Task: Write a Python function to calculate fibonacci num
Success: True | Latency: 431.04ms | Cost: $9e-05
---
06:22:07 | INFO | Trace recorded: 9ba3b261 | Latency: 636ms | Cost: $0.000143
Task: What are the SOLID principles in software engineer
Success: True | Latency: 636.21ms | Cost: $0.000143
---
06:22:08 | INFO | Trace recorded: 5bf11b80 | Latency: 1019ms | Cost: $0.000188
Task: Explain how HTTPS works
Success: True | Latency: 1019.32ms | Cost: $0.000188
---


Simulate Failed Call

In [5]:
print("========== SIMULATING FAILED CALL ==========\n")

tracer.trace(
    agent_name="ProductionAgent",
    task="Process payment transaction",
    response="",
    latency_ms=5500,
    tokens_used=2500,
    cost=0.015,
    success=False,
    error="Connection timeout after 5500ms"
)

print("Failed call simulated — check dashboard for alerts")

========== SIMULATING FAILED CALL ==========

06:22:08 | WARNING | ALERT: HIGH_LATENCY | Latency 5500ms exceeded threshold 3000ms
06:22:08 | WARNING | ALERT: HIGH_COST | Cost $0.015 exceeded threshold $0.01
06:22:08 | WARNING | ALERT: HIGH_TOKEN_USAGE | Tokens 2500 exceeded threshold 2000
06:22:08 | WARNING | ALERT: CALL_FAILED | Agent call failed: Connection timeout after 5500ms
06:22:08 | INFO | Trace recorded: 2918e2f9 | Latency: 5500ms | Cost: $0.015000
Failed call simulated — check dashboard for alerts


Dashboard

In [6]:
dashboard.render()


AGENTLENS DASHBOARD
Total Calls     : 6
Failed Calls    : 1
Success Rate    : 83.3%
Total Cost      : $0.01564
Total Tokens    : 3300
Avg Latency     : 1531.09ms
Max Latency     : 5500ms
Min Latency     : 431.04ms
Avg Cost/Call   : $0.002607
Total Alerts    : 4

--- ALERTS ---
  [HIGH_LATENCY] Latency 5500ms exceeded threshold 3000ms
  [HIGH_COST] Cost $0.015 exceeded threshold $0.01
  [HIGH_TOKEN_USAGE] Tokens 2500 exceeded threshold 2000
  [CALL_FAILED] Agent call failed: Connection timeout after 5500ms

--- TRACES ---
  ✅ [93f09371] ProductionAgent | 1154.97ms | $0.000178 | Explain the difference between SQL and N
  ✅ [7eb4377a] ProductionAgent | 431.04ms | $9e-05 | Write a Python function to calculate fib
  ✅ [9ba3b261] ProductionAgent | 636.21ms | $0.000143 | What are the SOLID principles in softwar
  ✅ [5bf11b80] ProductionAgent | 1019.32ms | $0.000188 | Explain how HTTPS works
  ❌ [2918e2f9] ProductionAgent | 5500ms | $0.015 | Process payment transaction


Detailed Trace Report

In [7]:
print("========== DETAILED TRACE REPORT ==========\n")
print(f"{'TraceID':<10} {'Agent':<18} {'Latency':>10} {'Tokens':>8} {'Cost':>10} {'Status'}")
print("-" * 70)
for trace in metrics_store.traces:
    status = "SUCCESS" if trace["success"] else "FAILED"
    print(f"{trace['trace_id']:<10} {trace['agent_name']:<18} {trace['latency_ms']:>8}ms {trace['tokens_used']:>8} ${trace['cost']:>9} {status}")

========== DETAILED TRACE REPORT ==========

TraceID    Agent                 Latency   Tokens       Cost Status
----------------------------------------------------------------------
cadbcb35   ProductionAgent       445.0ms       51 $  4.1e-05 SUCCESS
93f09371   ProductionAgent     1154.97ms      222 $ 0.000178 SUCCESS
7eb4377a   ProductionAgent      431.04ms      113 $    9e-05 SUCCESS
9ba3b261   ProductionAgent      636.21ms      179 $ 0.000143 SUCCESS
5bf11b80   ProductionAgent     1019.32ms      235 $ 0.000188 SUCCESS
2918e2f9   ProductionAgent        5500ms     2500 $    0.015 FAILED


Project Summary

In [8]:
print("========== AGENTLENS SUMMARY ==========\n")
print("Project      : AgentLens — Production Observability Agent")
print("Author       :K Murali Krishna")
print("Model        : Groq LLaMA-3.3-70b-versatile")
print("\nObservability Components:")
print("  ✓ MetricsStore  — stores all traces and aggregates")
print("  ✓ AlertManager  — fires alerts on threshold breach")
print("  ✓ Tracer        — intercepts and records every call")
print("  ✓ Dashboard     — renders real-time metrics view")
print("\nMetrics Tracked:")
print("  ✓ Latency per call (ms)")
print("  ✓ Token usage per call")
print("  ✓ Cost per call and total")
print("  ✓ Success/failure rate")
print("  ✓ Alert count and details")
print("\nAlert Types:")
print("  ✓ HIGH_LATENCY     — latency > 3000ms")
print("  ✓ HIGH_COST        — cost > $0.01 per call")
print("  ✓ HIGH_TOKEN_USAGE — tokens > 2000 per call")
print("  ✓ CALL_FAILED      — any exception")
print("\nProduction Concepts Demonstrated:")
print("  ✓ Full observability stack")
print("  ✓ Threshold-based alerting")
print("  ✓ Distributed tracing pattern")
print("  ✓ Production monitoring mindset")

========== AGENTLENS SUMMARY ==========

Project      : AgentLens — Production Observability Agent
Author       :K Murali Krishna
Model        : Groq LLaMA-3.3-70b-versatile

Observability Components:
  ✓ MetricsStore  — stores all traces and aggregates
  ✓ AlertManager  — fires alerts on threshold breach
  ✓ Tracer        — intercepts and records every call
  ✓ Dashboard     — renders real-time metrics view

Metrics Tracked:
  ✓ Latency per call (ms)
  ✓ Token usage per call
  ✓ Cost per call and total
  ✓ Success/failure rate
  ✓ Alert count and details

Alert Types:
  ✓ HIGH_LATENCY     — latency > 3000ms
  ✓ HIGH_COST        — cost > $0.01 per call
  ✓ HIGH_TOKEN_USAGE — tokens > 2000 per call
  ✓ CALL_FAILED      — any exception

Production Concepts Demonstrated:
  ✓ Full observability stack
  ✓ Threshold-based alerting
  ✓ Distributed tracing pattern
  ✓ Production monitoring mindset
